## Keys and imports

In [ ]:
!pip install python-dotenv langchain-openai langchain_community azure-core azure-search-documents==11.5.1 azure-storage-blob azure-identity openai aiohttp tiktoken --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.7/297.7 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.4/63.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.4/207.4 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.0/407.0 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 186.1/186.1 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.4/438.4 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.4/115.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.3 MB/s eta 0:00:00


In [1]:
storage_account_url = ""
storage_account_key = ""

embedding_endpoint = ""

AZURE_SEARCH_SERVICE: str = ""
AZURE_SEARCH_KEY: str = ""
AZURE_OPENAI_ACCOUNT: str = ""
AZURE_OPENAI_KEY: str = ""
AZURE_AI_MULTISERVICE_ACCOUNT: str = ""
AZURE_AI_MULTISERVICE_KEY: str = ""
AZURE_STORAGE_CONNECTION: str = ""

from azure.core.credentials import AzureKeyCredential
AZURE_SEARCH_CREDENTIAL = AzureKeyCredential(AZURE_SEARCH_KEY)

from aiohttp import ClientSession
import pandas as pd
import asyncio
import nest_asyncio
from tqdm import tqdm
import numpy as np

import os
import openai

os.environ["AZURE_OPENAI_ENDPOINT"] = ""
os.environ["AZURE_OPENAI_API_KEY"] = ""
os.environ["SEARCH_ENDPOINT"] = ""
os.environ["SEARCH_KEY"] = ""
os.environ["SEARCH_INDEX_NAME"] = ""
os.environ["EMBEDDING_ENDPOINT"] = ""
os.environ["EMBEDDING_KEY"] = ""

## Indexing - ONLY RUN ONCE

In [ ]:
import os
import requests

# Create the 'guidelines' directory if it doesn't exist
if not os.path.exists('guidelines'):
    os.makedirs('guidelines')

def download_file(url, filename):
    try:
        response = requests.get(url, stream=True, headers={'User-Agent': 'Mozilla/5.0'})  # Add a User-Agent header
        response.raise_for_status()  # Raise an exception for bad status codes (4xx or 5xx)

        with open(os.path.join('guidelines', filename), 'wb') as file:
            for chunk in response.iter_content(chunk_size=8192):
                file.write(chunk)
        print(f"Downloaded {filename} successfully.")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading {filename}: {e}")


url1 = 'https://www.abim.org/Media/bfijryql/laboratory-reference-ranges.pdf'
download_file(url1, 'laboratory-reference-ranges.pdf')

Downloaded laboratory-reference-ranges.pdf successfully.


In [ ]:
textpaths = []
for root, dirs, files in os.walk("./guidelines"):
  for file in files:
    textpaths.append(os.path.join(root, file))

In [ ]:
from azure.storage.blob import BlobServiceClient, ContentSettings

blob_service_client = BlobServiceClient(storage_account_url, storage_account_key)

container_client = blob_service_client.create_container(name='smallchunk-abim-2025')

In [ ]:
for text in textpaths:
  snippet = text.split('.pdf')[0].split('/')[-1] # Extract only filename to avoid invalid blob names
  print(snippet)
  blob_client = blob_service_client.get_blob_client(container="smallchunk-abim-2025", blob=snippet)

  # Set content type to application/pdf for PDF files
  content_settings = ContentSettings(content_type='application/pdf')

  with open(text, "rb") as data:
      blob_client.upload_blob(data, overwrite=True, content_settings=content_settings)

laboratory-reference-ranges


In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchField,
    SearchFieldDataType,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    SearchIndex
)

pre_name = 'smallchunk-abim-2025'

# Create a search index
index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE, credential=AZURE_SEARCH_CREDENTIAL)
fields = [
    SearchField(name="parent_id", type=SearchFieldDataType.String),
    SearchField(name="title", type=SearchFieldDataType.String),
    SearchField(name="chunk_id", type=SearchFieldDataType.String, key=True, sortable=True, filterable=True, facetable=True, analyzer_name="keyword"),
    SearchField(name="content", type=SearchFieldDataType.String, sortable=False, filterable=False, facetable=False),
    SearchField(name="content_vector", type=SearchFieldDataType.Collection(SearchFieldDataType.Single), vector_search_dimensions=3072, vector_search_profile_name="myHnswProfile")
    ]

# Configure the vector search configuration
vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(name="myHnsw"),
    ],
    profiles=[
        VectorSearchProfile(
            name="myHnswProfile",
            algorithm_configuration_name="myHnsw",
            vectorizer_name="myOpenAI",
        )
    ],
    vectorizers=[
        AzureOpenAIVectorizer(
            vectorizer_name="myOpenAI",
            kind="azureOpenAI",
            parameters=AzureOpenAIVectorizerParameters(
                resource_url=AZURE_OPENAI_ACCOUNT,
                deployment_name="text-embedding-3-large",
                model_name="text-embedding-3-large",
                api_key=AZURE_OPENAI_KEY
            ),
        ),
    ],
)

In [ ]:
# Create the search index
index = SearchIndex(name=pre_name, fields=fields, vector_search=vector_search)
result = index_client.create_or_update_index(index)
print(f"Index '{result.name}' created or updated")

Index 'smallchunk-abim-2025' created or updated


In [ ]:
from azure.search.documents.indexes import SearchIndexerClient
from azure.search.documents.indexes.models import (
    SearchIndexerDataContainer,
    SearchIndexerDataSourceConnection
)

# Create a data source
indexer_client = SearchIndexerClient(endpoint=AZURE_SEARCH_SERVICE, credential=AZURE_SEARCH_CREDENTIAL)
container = SearchIndexerDataContainer(name=pre_name)
data_source_connection = SearchIndexerDataSourceConnection(
    name = pre_name,
    type="azureblob",
    connection_string=AZURE_STORAGE_CONNECTION,
    container=container
)
data_source = indexer_client.create_or_update_data_source_connection(data_source_connection)

print(f"Data source '{data_source.name}' created or updated")

Data source 'smallchunk-abim-2025' created or updated


In [ ]:
from azure.search.documents.indexes.models import (
    SplitSkill,
    InputFieldMappingEntry,
    OutputFieldMappingEntry,
    AzureOpenAIEmbeddingSkill,
    EntityRecognitionSkill,
    SearchIndexerIndexProjection,
    SearchIndexerIndexProjectionSelector,
    SearchIndexerIndexProjectionsParameters,
    IndexProjectionMode,
    SearchIndexerSkillset,
    CognitiveServicesAccountKey
)

# Create a skillset

split_skill = SplitSkill(
    description="Split skill to chunk documents",
    text_split_mode="pages",
    context="/document",
    maximum_page_length=800,
    page_overlap_length=100,
    inputs=[
        InputFieldMappingEntry(name="text", source="/document/content"),
    ],
    outputs=[
        OutputFieldMappingEntry(name="textItems", target_name="pages")
    ],
)

embedding_skill = AzureOpenAIEmbeddingSkill(
    description="Skill to generate embeddings via Azure OpenAI",
    context="/document/pages/*",
    resource_url=AZURE_OPENAI_ACCOUNT,
    deployment_name="text-embedding-3-large",
    model_name="text-embedding-3-large",
    dimensions=3072,
    inputs=[
        InputFieldMappingEntry(name="text", source="/document/pages/*"),
    ],
    outputs=[
        OutputFieldMappingEntry(name="embedding", target_name="text_vector")
    ],
)

index_projections = SearchIndexerIndexProjection(
    selectors=[
        SearchIndexerIndexProjectionSelector(
            target_index_name=pre_name,
            parent_key_field_name="parent_id",
            source_context="/document/pages/*",
            mappings=[
                InputFieldMappingEntry(name="content", source="/document/pages/*"),
                InputFieldMappingEntry(name="content_vector", source="/document/pages/*/text_vector"),
                InputFieldMappingEntry(name="title", source="/document/metadata_storage_name"),
            ],
        ),
    ],
    parameters=SearchIndexerIndexProjectionsParameters(
        projection_mode=IndexProjectionMode.SKIP_INDEXING_PARENT_DOCUMENTS
    ),
)

cognitive_services_account = CognitiveServicesAccountKey(key=AZURE_AI_MULTISERVICE_KEY)

skills = [split_skill, embedding_skill]

skillset = SearchIndexerSkillset(
    name=pre_name,
    description="Skillset to chunk documents and generating embeddings",
    skills=skills,
    index_projection=index_projections,
    cognitive_services_account=cognitive_services_account
)

indexer_client.create_or_update_skillset(skillset)
print(f"Skillset '{skillset.name}' created or updated")

Skillset 'smallchunk-abim-2025' created or updated


In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndexer,
    FieldMapping
)

# Create an indexer

indexer_parameters = None

indexer = SearchIndexer(
    name=pre_name,
    description="Indexer to index documents and generate embeddings",
    skillset_name=pre_name,
    target_index_name=pre_name,
    data_source_name=pre_name,
    parameters=indexer_parameters
)

# Create and run the indexer
indexer_result = indexer_client.create_or_update_indexer(indexer)

print(f"Indexer '{indexer.name}' is created and running. Give the indexer a few minutes before running a query.")

Indexer 'smallchunk-abim-2025' is created and running. Give the indexer a few minutes before running a query.


## Prompt and input

In [ ]:
rag_prompt_no_example = '''You are an expert diagnostician machine for use by doctors. If the user input is not patient data, you politely decline the request. Please suggest diagnoses and conditions, followed by the evidence points supporting each diagnosis in the form of bullet points. Include previous diagnoses and pertinent information about the patient's medical history (if any). Pay close attention to all the history and investigations provided.  Put asterisks around the diagnoses to highlight them. Give each evidence points as a separate bullet point beneath the diagnosis. Include in your evidence points any relevant clinical scores that can be calculated from the information I have given. Do not explain the evidence points, only state them. For every diagnosis you list, if there are alternative differentials possible, state the most likely three in a bullet point beneath the evidence points (you do not need to state the evidence supporting them - you only need to do that for the main diagnoses). For the main diagnoses, give only confirmed diagnoses and evidence points that can be inferred solely based on the information I have given - do not use any other information. Only give me the information I have asked for - do not give me any other information. Do not give me any introductions or conclusions, safety instructions, or safety warnings. Use British English. This system is Retrieval-Augmented Generation (RAG) enabled. **Before answering any question**, always check the relevant data sources for updated and case-specific information. Ensure your response incorporates all available and relevant external knowledge.

                                To illustrate how the information should be presented:

                                *MAIN DIAGNOSIS 1 AS HEADING*
                                evidence points to support MAIN DIAGNOSIS 1
                                The final bullet point is alternative differentials to consider: alternative 1, alternative 2, alternative 3

                                *MAIN DIAGNOSIS 2 AS HEADING*
                                evidence points to support MAIN DIAGNOSIS 2
                                The final bullet point is alternative differentials to consider: alternative 1, alternative 2, alternative 3

                                and so on...

Before finalising your answer check if you haven't missed any abnormal data points and hence any diagnoses or alternative differentials that could be made based on them. If you did, add them to your reply. If two diagnoses are commonly caused by the same underlying disease, have them under one header, which is the underlying disease.
'''

In [ ]:
follow_up_message = """Below are the actual diagnoses of the same patient reported by clinicians.
Go through the actual diagnoses and cross-check each actual diagnosis with \
the initial list of diagnoses you provided answer the following two questions:

Question 1: Is this actual diagnosis a new disease, not directly related to any of the diagnoses or alternatives you suggested \
in your initial list? If an actual diagnosis is a complication of, a more specific version of, or falls under a broader \
category of a diagnosis / alternative you initially listed, it should not be considered a new disease. If an actual diagnosis \
affects the same organ as a diagnosis / alternative you initially listed, but it has a different onset and progression \
(for example, the actual diagnosis is chronic but you initially listed the acute disease), then your answer should be 'No'. \
If an actual diagnosis is caused by the same pathogen as a diagnosis in your initial list, the answer should also be 'No'. \
If an actual diagnosis is not a medical diagnosis, your answer should be 'No'.

If your answer to Question 1 was 'No', put N/A as answer for Question 2 and skip to the Example below. If your answer to Question 1 was 'Yes', always answer Question 2!

Question 2: Would it be possible to directly infer this actual diagnosis from the patient data provided in the initial query?
If yes, support with facts: quote exact numbers or text from the initial query.
If no, in case the data contradicts the diagnosis, quote the data and say why it does not support the diagnosis. \
Otherwise, please specify what additional data would have been helpful to establish this diagnosis.

Example:
If the patient data is:
"Blood report: min potassium: 3.1, avg hemoglobin: 14.5, max sodium: 139, avg wbc: 13.9
Blood gas report: ph: 7.2
Imaging report: patient with polysubstance abuse, lungs look normal"

and your initial list in your previous response contained the following suggested diagnoses:
*Acidosis*
- ph of 7.2
- Alternative differentials to consider: respiratory acidosis, metabolic acidosis, mixed acid-base disorder

*Polysubstance abuse*
- The imaging report mentions "patient with polysubstance abuse"
- Alternative differentials to consider: alcohol abuse, drug abuse, signs of withdrawal'

*Leukocytosis*
- avg wbc of 13.9
- Alternative differentials to consider: infection, inflammatory condition, myeloproliferative disorder

and actual diagnoses are:
D1: Poisoning by cocaine
D2: Hypokalemia
D3: Hypernatremia
D4: Severe sepsis

Then your answer should be:
D1: Poisoning by cocaine
Question 1: No, this is similar to diagnosis *Polysubstance abuse*
Question 2: N/A

D2: Hypokalemia
Question 1: Yes
Question 2: Yes, the blood report mentions "min potassium: 3.1"

D3: Hypernatremia
Question 1: Yes
Question 2: No, the blood report mentions "max sodium: 139", but only sodium levels above 145 mmol/L indicate hypernatremia, \
hence the data does not support hypernatremia.

D4: Severe sepsis
Question 1: Yes
Question 2: No, additional data such as fever, increased heart rate, increased respiratory rate, positive blood cultures, or evidence of organ dysfunction would have been helpful to establish this diagnosis. "

Before finalizing your answer check if you haven't missed noticing any diagnoses from your initial list that are related to \
any of the actual diagnoses you answered the two questions for! If you did, modify the answers to the questions accordingly!

Actual diagnoses:\n"""

In [ ]:
result_df = pd.read_csv('MIMIC-IV_input.csv', index_col = 0)

## Diagnostic predictions

In [ ]:
nest_asyncio.apply()

In [ ]:
from openai import AsyncAzureOpenAI
from azure.search.documents import SearchClient
#from langchain.vectorstores.azuresearch import AzureSearch
import langchain_community.vectorstores.azuresearch as azuresearch
from langchain_openai import AzureOpenAIEmbeddings

import os #needed for AzureOpenAIEmbeddings

# Set up clients and specify the chat model
openai_client = AsyncAzureOpenAI(
     api_version="2025-01-01-preview",
     azure_endpoint=AZURE_OPENAI_ACCOUNT,
     api_key=AZURE_OPENAI_KEY
 )

deployment_name = "gpt-4o-05-13"

def embedding_func():
    return AzureOpenAIEmbeddings(
        azure_deployment="text-embedding-3-large",
        chunk_size=20000  #larger than a reasonable input
    )

search = azuresearch.AzureSearch(
    azure_search_endpoint=os.environ["SEARCH_ENDPOINT"],
    azure_search_key=os.environ["SEARCH_KEY"],
    index_name='smallchunk-abim-2025',
    embedding_function=embedding_func(),
    additional_search_client_options={"retry_total": 5},
)

In [ ]:
async def get_completion(query):
  results = await search.ahybrid_search(query = query, k=10)

  sources_formatted = "\n".join([f'TITLE: {document.metadata["title"]}, CONTENT: {document.page_content}' for document in results])

  USER_PROMPT="""
  Patient data:\n{query}
  Sources:\n{sources}"""

  response = await openai_client.chat.completions.create(
      messages=[
          {
              "role": "system",
              "content": rag_prompt_no_example, #global
          },
          {
              "role": "user",
              "content": USER_PROMPT.format(query=query, sources=sources_formatted)
          },
      ],
      model=deployment_name
  )

  return response

In [ ]:
async def get_diagnoses(queries):
    loop = asyncio.get_event_loop()
    tasks = []
    for query in queries:
        tasks.append(get_completion(query))
    all_data = loop.run_until_complete(asyncio.gather(*tasks))
    return all_data

In [ ]:
result_df['GPT-Diagnoses'] = result_df['GPT-Diagnoses'].astype('str')
result_df['GPT-Eval'] = result_df['GPT-Eval'].astype('str')

query_per_call = 4
repeat = []
for i in tqdm(range(0,1000,query_per_call)):
    try:
        idx_from = i
        idx_to = i + query_per_call
        queries = [result_df.iloc[i]['GPT_input'] for i in range(idx_from,idx_to)]
        result = asyncio.run(get_diagnoses(queries))
        contents = [result[i].choices[0].message.content for i in range(query_per_call)]
        hadm_ids = [result_df.index[i] for i in range(idx_from,idx_to)]
        result_df.loc[hadm_ids, 'GPT-Diagnoses'] = contents
    except Exception as e:
        print('Error happened at iteration i: ' + str(i))
        repeat.append(i)
        print(e)

100%|██████████| 250/250 [38:12<00:00,  9.17s/it]


In [ ]:
for i in tqdm(repeat):
    try:
        idx_from = i
        idx_to = i + query_per_call
        queries = [result_df.iloc[i]['GPT_input'] for i in range(idx_from,idx_to)]
        result = asyncio.run(get_diagnoses(queries))
        contents = [result[i].choices[0].message.content for i in range(query_per_call)]
        hadm_ids = [result_df.index[i] for i in range(idx_from,idx_to)]
        result_df.loc[hadm_ids, 'GPT-Diagnoses'] = contents
    except Exception as e:
        print('Error happened at iteration i: ' + str(i))
        print(e)

assert result_df['GPT-Diagnoses'].isna().sum() == 0

0it [00:00, ?it/s]


## LLM-as-a-judge Eval

In [ ]:
result_df = pd.read_csv('run1-smallchunk-41eval-ABIM2025-GPT4oRAGProd.csv', index_col=0)
result_df['GPT-Eval']= ''

In [ ]:
async def get_followup(query, response, follow_up):

    patient_data = query.split('Patient data:\n')[1]

    results = await search.ahybrid_search(query = patient_data, k=10)

    sources_formatted = "\n".join([f'TITLE: {document.metadata["title"]}, CONTENT: {document.page_content}' for document in results])

    SOURCES_PROMPT="""
    Sources:\n{sources}"""

    RAG_FOLLOW_UP="""
    This system is Retrieval-Augmented Generation (RAG) enabled. **Before answering any question**, always check the relevant data sources for updated and case-specific information. Ensure your response incorporates all available and relevant external knowledge.
    {follow_up}"""

    response = await openai_client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant who gives reasons for all answers."
            },
            {
                "role": "user",
                "content": query
            },
            {
                "role": "assistant",
                "content": response
            },
            {
                "role": "user",
                "content": RAG_FOLLOW_UP.format(follow_up=follow_up) + SOURCES_PROMPT.format(sources=sources_formatted)
            }
        ],
        model="gpt-4.1",
        temperature = 0
    )

    # response = await openai_client.chat.completions.create(
    #     model="gpt-4.1",
    #     temperature = 0,
    #     messages=[
    #         {"role": "system", "content": "You are a helpful diagnostic assistant who gives reasons for all answers."},
    #         {"role": "user", "content": query},
    #         {"role": "assistant", "content": response},
    #         {"role": "user", "content": follow_up}
    #     ])
    return response#.choices[0].message["content"]

async def get_evaluation(queries, responses, follow_ups):
    loop = asyncio.get_event_loop()
    tasks = []
    for query, response, follow_up in zip(queries, responses, follow_ups):
        tasks.append(get_followup(query, response, follow_up))
    all_data = loop.run_until_complete(asyncio.gather(*tasks))
    return all_data

query_per_call = 8
repeat = []
for i in tqdm(range(0,1000,query_per_call)):
    try:
        idx_from = i
        idx_to = i + query_per_call
        queries = [rag_prompt_no_example + 'Patient data:\n' + result_df.iloc[i]['GPT_input'] for i in range(idx_from,idx_to)]
        responses = result_df.iloc[idx_from:idx_to,:]['GPT-Diagnoses']
        follow_ups = [follow_up_message + result_df.iloc[i]['diagnoses'].replace('\n', '\nD').replace('1:', 'D1:', 1) for i in range(idx_from,idx_to)]
        result = asyncio.run(get_evaluation(queries, responses, follow_ups))
        contents = [result[i].choices[0].message.content for i in range(query_per_call)]
        hadm_ids = [result_df.index[i] for i in range(idx_from,idx_to)]
        result_df.loc[hadm_ids, 'GPT-Eval'] = contents
    except Exception as e:
        print('Error happened at iteration i: ' + str(i))
        repeat.append(i)
        print(e)

 26%|██▌       | 32/125 [14:06<49:02, 31.64s/it]ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-3505' coro=<get_evaluation() done, defined at <ipython-input-13-ddcca2d2aff2>:13> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "<ipython-input-13-ddcca2d2aff2>", line 30, in <cell line: 0>
    result = asyncio.run(get_evaluation(queries, responses, follow_ups))
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/nest_asyncio.py", line 30, in run
    return loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/nest_asyncio.py", line 92, in run_until_complete
    self._run_once()
  File "/usr/local/lib/python3.11/dist-packages/nest_asyncio.py"

In [ ]:
for i in tqdm(repeat):
    try:
        idx_from = i
        idx_to = i + query_per_call
        queries = [rag_prompt_no_example + 'Patient data:\n' + result_df.iloc[i]['GPT_input'] for i in range(idx_from,idx_to)]
        responses = result_df.iloc[idx_from:idx_to,:]['GPT-Diagnoses']
        follow_ups = [follow_up_message + result_df.iloc[i]['diagnoses'].replace('\n', '\nD').replace('1:', 'D1:', 1) for i in range(idx_from,idx_to)]
        result = asyncio.run(get_evaluation(queries, responses, follow_ups))
        contents = [result[i].choices[0].message.content for i in range(query_per_call)]
        hadm_ids = [result_df.index[i] for i in range(idx_from,idx_to)]
        result_df.loc[hadm_ids, 'GPT-Eval'] = contents
    except Exception as e:
        print('Error happened at iteration i: ' + str(i))
        print(e)

0it [00:00, ?it/s]


In [ ]:
result_df[result_df['GPT-Eval'].isna()]

,hadm_id,diagnoses,GPT_input,GPT-Diagnoses,GPT-Eval


In [ ]:
name = 'run1-smallchunk-RAG41eval-ABIM2025-GPT4oRAGProd.csv'

result_df.to_csv(name)

from google.colab import files
import time

time.sleep(5)
files.download(name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
result_df = pd.read_csv('run1-smallchunk-RAG41eval-ABIM2025-GPT4oRAGProd.csv', index_col = 0)

## Analysis

In [3]:
import re

def analyze_results(text, index):
    mistakes = []
    hits = []
    excluded = [] #not a medical diagnosis
    noninferables = []
    current = 1
    total_adjust = 0
    #for conditions that GPT-4 grouped together - still doesn't capture issue with hadmid 23707730
    from_nums = []
    to_nums = []
    grouped = re.findall(r'\n\d+-\d+:', text)
    if len(grouped) > 0:
        #print("Grouping found!")
        #print("At index: ")
        #print(index)
        for elem in grouped:
            from_nums.append(str(int(elem.split('-')[0]))) #str(int()) for safety
            to_nums.append(str(int(elem.split('-')[1].strip(':'))))
    while 1:
        try:
            if current == 1:
                number = str(current)
                #sometimes GPT-4 adds words like Diagnosis or Actual diagnosis, and we want to capture that
                pre_word = text.split(number, 1)[0]
            #for conditions that GPT-4 grouped together
            elif str(current) in from_nums:
                idx = from_nums.index(str(current))
                number = grouped[idx]
                total_adjust += int(to_nums[idx]) - current
                current = int(to_nums[idx])
            else:
                number = '\n' + pre_word + str(current)
            nextOne = '\n' + pre_word + str(current+1)
            if text.split(number, 1)[1].split('Question 1: ', 1)[1][:2] == 'No':
                if 'not a medical diagnosis' in text.split(number, 1)[1].split('Question 1: ', 1)[1].split('Question 2: ', 1)[0].split(nextOne, 1)[0]:
                    print(index)
                    print(text.split(number, 1)[1].split('Question 1: ', 1)[0])
                    excluded.append(str(current))
                else:
                    hits.append(str(current))
            elif text.split(number, 1)[1].split('Question 2: ', 1)[1][:3] == 'Yes':
                mistakes.append(str(current))
            elif text.split(number, 1)[1].split('Question 2: ', 1)[1][:2] == 'No':
                noninferables.append(str(current))
            else:
                print("Unable to parse text when looking at diagnosis number: ")
                print(current)
                print("At index: ")
                print(index)
        except:
            #print('Diagnosis number not found in text: ')
            #print(current)
            total = current - 1 - total_adjust
            break
        current += 1
    return pd.Series([len(hits), len(noninferables), len(mistakes), len(excluded), '; '.join(hits), '; '.join(noninferables), '; '.join(mistakes), '; '.join(excluded), total])

In [4]:
analyzed_df = result_df.apply(lambda row: analyze_results(row['GPT-Eval'], row.name),1)
analyzed_df.columns = ['no_hits', 'no_noninferables', 'no_mistakes', 'no_excluded', 'hits', 'noninferables', 'mistakes', 'excluded', 'total_ICD_diagnoses']
analyzed_df['error'] = analyzed_df['no_mistakes'] / (analyzed_df['no_hits'] + analyzed_df['no_mistakes'])
analyzed_df['sensitivity'] = 1-analyzed_df['error']
print(analyzed_df['sensitivity'].mean())
print(1-(analyzed_df['no_mistakes'].sum() / (analyzed_df['no_hits'].sum() + analyzed_df['no_mistakes'].sum())))

results = pd.concat([result_df, analyzed_df], axis=1)

0
: Long-term (current) use of aspirin  

6
: Encounter for examination for normal comparison and control in clinical research program  

8
: Physical restraint status  

9
: Other dependence on machines, supplemental oxygen  

14
: Unspecified place or not applicable  

16
: Other place in single-family (private) house as the place of occurrence of the external cause  

16
: Physical restraint status  

16
: Family history of alcohol abuse and dependence  

16
: Family history of other mental and behavioral disorders  

16
: Exposure to other specified factors, initial encounter  

16
: Unspecified place or not applicable  

16
: Tobacco use  

19
: Examination of participant in clinical trial  

19
: Accidents occurring in residential institution  

21
: Long term (current) use of antithrombotics/antiplatelets  

21
: Personal history of nicotine dependence  

22
: Encounter for examination for normal comparison and control in clinical research program  

24
: Other specified places 

In [5]:
results['no_hits'].sum()

np.int64(5069)

In [6]:
results['no_hits'].sum() + results['no_mistakes'].sum()

np.int64(5500)

In [7]:
print(results['no_hits'].sum() + results['no_mistakes'].sum())
print(results['no_excluded'].sum() + results['no_noninferables'].sum())
print(results['no_hits'].sum() + results['no_mistakes'].sum() + results['no_excluded'].sum() + results['no_noninferables'].sum())

5500
8903
14403


In [8]:
def get_identified_diagnoses(text, numbers):
    hit_names = []
    if numbers == '' or numbers == 'nan':
        return hit_names
    nums = numbers.split(';')
    names = text.split('\n')
    try:
        for num in nums:
            diag = names[int(num.strip())-1]
            name = re.sub('[0-9]+:', '', diag, 1)
            hit_names.append(name)
    except ValueError:
        print(nums)
    return hit_names

results['hits'] = results['hits'].astype('str')
results['hit_names'] = results.apply(lambda row: get_identified_diagnoses(row['diagnoses'], row['hits']),1)

results['mistakes'] = results['mistakes'].astype('str')
results['mistake_names'] = results.apply(lambda row: get_identified_diagnoses(row['diagnoses'], row['mistakes']),1)

results['noninferables'] = results['noninferables'].astype('str')
results['noninferable_names'] = results.apply(lambda row: get_identified_diagnoses(row['diagnoses'], row['noninferables']),1)

In [9]:
results['hit_names'].apply(len).sum()

np.int64(5069)

In [10]:
all_hits = []
for i in results['hit_names'].values:
    for hits in i:
        all_hits.append(hits)

all_mistakes = []
for i in results['mistake_names'].values:
    for mistakes in i:
        all_mistakes.append(mistakes)

In [11]:
pd.DataFrame(all_mistakes).value_counts().head(20)

,count
0,
"Diabetes mellitus without mention of complication, type II or unspecified type, not stated as uncontrolled",30
"Acute kidney failure, unspecified",22
Hypoxemia,18
Unspecified essential hypertension,12
Hyperkalemia,10
Acute respiratory failure with hypoxia,9
Hypokalemia,9
Other iatrogenic hypotension,9
"Thrombocytopenia, unspecified",9


In [12]:
df = pd.DataFrame(all_mistakes).value_counts().reset_index()
df.columns = ['name', 'count']
df.loc[df['name'].str.contains('osmo'),:]

,name,count
16,Hyperosmolality and hypernatremia,7
17,Hypo-osmolality and hyponatremia,7
22,Hyposmolality and/or hyponatremia,5
33,Hyperosmolality and/or hypernatremia,3


In [13]:
df.loc[df['name'].str.contains('osmo'),:].sum()

,0
name,Hyperosmolality and hypernatremiaHypo-osmolali...
count,22


In [14]:
df = pd.DataFrame(all_hits).value_counts().reset_index()
df.columns = ['name', 'count']
df.loc[df['name'].str.contains('osmo'),:]

,name,count
25,Hyposmolality and/or hyponatremia,26
52,Hypo-osmolality and hyponatremia,18
226,Hyperosmolality and/or hypernatremia,4
561,Type 2 diabetes mellitus with hyperosmolarity ...,2
663,Hyperosmolality and hypernatremia,1
838,"Diabetes with hyperosmolarity, type II or unsp...",1
869,"Diabetes with hyperosmolarity, type II or unsp...",1


In [15]:
df.loc[df['name'].str.contains('osmo'),:].sum()

,0
name,Hyposmolality and/or hyponatremiaHypo-osmolali...
count,53
